In [ ]:
# 1. 安装 UMAP
!pip install umap-learn

# 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. 在你的云盘里创建一个专门的实验文件夹
import os
# 我们将所有数据和图表保存在这个目录，哪怕断线也不会丢失
WORK_DIR = '/content/drive/MyDrive/Denoise_Experiment'
os.makedirs(WORK_DIR, exist_ok=True)
print(f"工作目录已准备好: {WORK_DIR}")

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import MDS, Isomap, TSNE
import umap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import time

# 确认工作目录 (继承上一个单元格)
WORK_DIR = '/content/drive/MyDrive/Denoise_Experiment'
data_dir = os.path.join(WORK_DIR, 'data')

In [ ]:
# ==========================================
# 1. 数据加载与预处理
# ==========================================
print("正在加载 CIFAR-10 基础图像数据...")
transform = transforms.Compose([transforms.ToTensor()])
# 数据下载到云盘，下次运行就不用重新下载了
trainset = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform)

# 提取图像并展平为一维向量 (N, 3072)
X_all = trainset.data.reshape(trainset.data.shape[0], -1)
X_all = X_all / 255.0

# ==========================================
# 2. 自动下载并加载 CIFAR-10N 真实人类噪声标签
# ==========================================
print("正在检查并加载 CIFAR-10N 噪声标签文件...")
url = "https://github.com/UCSC-REAL/cifar-10-100n/raw/main/data/CIFAR-10_human.pt"
file_path = os.path.join(data_dir, "CIFAR-10_human.pt")

if not os.path.exists(file_path):
    print(f"本地未找到，正在从 {url} 下载，请稍候...")
    os.makedirs(data_dir, exist_ok=True)
    urllib.request.urlretrieve(url, file_path)
    print("下载完成！")
else:
    print("文件已存在云盘，直接读取。")

# 加载真实人类噪声文件 (weights_only=False 解决安全拦截)
noise_file = torch.load(file_path, weights_only=False)

# 使用 'worse_label' 作为测试基准
y_noisy_all = np.array(noise_file['worse_label'])
y_clean_all = np.array(trainset.targets)

In [ ]:

# ==========================================
# 3. 采样数据 (防止内存爆炸)
# ==========================================
N_SAMPLES = 3000
np.random.seed(42)
indices = np.random.choice(len(X_all), N_SAMPLES, replace=False)
X = X_all[indices]
y_clean = y_clean_all[indices]
y_noisy = y_noisy_all[indices]
print(f"数据采样完毕，即将进行降维对比: 样本数={X.shape[0]}, 维度={X.shape[1]}")

# ==========================================
# 4. 定义降维算法 & 5. 运行评估
# ==========================================
reducers = {
    "PCA": PCA(n_components=2),
    "KPCA (RBF)": KernelPCA(n_components=2, kernel='rbf', fit_inverse_transform=True),
    "MDS": MDS(n_components=2, normalized_stress='auto'),
    "ISOMAP": Isomap(n_components=2, n_neighbors=10),
    "t-SNE": TSNE(n_components=2, perplexity=30, random_state=42),
    "UMAP": umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
}

plt.figure(figsize=(20, 10))
results = {}

for i, (name, model) in enumerate(reducers.items()):
    print(f"正在运行 {name} ...")
    start_time = time.time()

    X_2d = model.fit_transform(X)
    t_cost = time.time() - start_time

    # KNN 提纯去噪评估
    knn = KNeighborsClassifier(n_neighbors=10)
    knn.fit(X_2d, y_noisy)
    y_pred_corrected = knn.predict(X_2d)

    denoise_acc = accuracy_score(y_clean, y_pred_corrected)
    results[name] = {'acc': denoise_acc, 'time': t_cost}

    # 可视化绘图
    plt.subplot(2, 3, i + 1)
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_clean, cmap='tab10', s=5, alpha=0.7)
    plt.title(f"{name}\nDenoise Acc: {denoise_acc:.2%} | Time: {t_cost:.1f}s", fontsize=12)
    plt.xticks([]); plt.yticks([])

plt.tight_layout()
# 将高清结果图直接保存到你的云盘，方便写报告时使用
save_path = os.path.join(WORK_DIR, 'manifold_comparison.png')
plt.savefig(save_path, dpi=300)
plt.show()

print("\n=== 去噪提纯准确率对比 ===")
for name, metrics in results.items():
    print(f"{name:10s} | 准确率: {metrics['acc']:.2%} | 耗时: {metrics['time']:.2f}s")
print(f"\n✅ 结果图已安全保存在云盘: {save_path}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from datasets import load_dataset
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm

# ==========================================
# 1. 准备 ImageNet-100 数据集 (取子集用于快速验证)
# ==========================================
print("正在处理 ImageNet-100 数据集...")
# 从刚才下载的高速缓存中读取
dataset = load_dataset("clane9/imagenet-100", cache_dir="/content/imagenet100_cache")

# 定义图像预处理 (缩放到 128x128，转换为 Tensor)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # 归一化到 [-1, 1]
])

# 封装为 PyTorch Dataset
class ImageNetSubset(Dataset):
    def __init__(self, hf_dataset, num_samples=1000):
        self.dataset = hf_dataset['train'].select(range(num_samples))
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img = self.dataset[idx]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return transform(img)

# 仅取前 1000 张图进行概念验证训练 (节约时间)
train_subset = ImageNetSubset(dataset, num_samples=1000)
dataloader = DataLoader(train_subset, batch_size=32, shuffle=True)

# ==========================================
# 2. 定义前向加噪函数 (扩散过程)
# ==========================================
def q_sample(x_start, t, noise=None):
    """
    根据时间步 t 加噪: x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise
    (这里做了极简化的线性假设用于概念演示)
    """
    if noise is None:
        noise = torch.randn_like(x_start)

    # 模拟 alpha_bar (随着 t 增大，原图比例减小，噪声比例增加)
    # t 在 [0, 1] 之间
    alpha_bar = (1.0 - t).view(-1, 1, 1, 1)

    x_t = torch.sqrt(alpha_bar) * x_start + torch.sqrt(1 - alpha_bar) * noise
    return x_t, noise

# ==========================================
# 3. 初始化两个并行的 JiT 模型
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"训练设备: {device}")

# 复用上一节定义的 JiT_Model
# 模型 A: 用于预测纯噪声 (\epsilon-prediction)
model_eps = JiT_Model(img_size=128, patch_size=16, embed_dim=256, depth=4, num_heads=4).to(device)
# 模型 B: 用于预测干净原图 (x-prediction - 何恺明方法)
model_x = JiT_Model(img_size=128, patch_size=16, embed_dim=256, depth=4, num_heads=4).to(device)

optimizer_eps = optim.AdamW(model_eps.parameters(), lr=1e-4)
optimizer_x = optim.AdamW(model_x.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ==========================================
# 4. 开始对比训练循环
# ==========================================
EPOCHS = 10
loss_history_eps = []
loss_history_x = []

print("\n🚀 开始对比训练：ε-预测 vs x-预测")
for epoch in range(EPOCHS):
    epoch_loss_eps = 0
    epoch_loss_x = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_x0 in progress_bar:
        batch_x0 = batch_x0.to(device)
        B = batch_x0.shape[0]

        # 随机采样时间步 t [0, 1]
        t = torch.rand(B, 1).to(device)

        # 生成真实噪声
        real_noise = torch.randn_like(batch_x0)

        # 加噪得到 x_t
        batch_xt, _ = q_sample(batch_x0, t, noise=real_noise)

        # ----------------------------------
        # 模式一：ε-预测 (预测噪声)
        # ----------------------------------
        optimizer_eps.zero_grad()
        pred_eps = model_eps(batch_xt, t)
        loss_eps = criterion(pred_eps, real_noise) # 目标是纯噪声
        loss_eps.backward()
        optimizer_eps.step()
        epoch_loss_eps += loss_eps.item()

        # ----------------------------------
        # 模式二：x-预测 (预测原图)
        # ----------------------------------
        optimizer_x.zero_grad()
        pred_x0 = model_x(batch_xt, t)
        loss_x = criterion(pred_x0, batch_x0) # 目标是干净原图
        loss_x.backward()
        optimizer_x.step()
        epoch_loss_x += loss_x.item()

        progress_bar.set_postfix({"L_eps": f"{loss_eps.item():.4f}", "L_x": f"{loss_x.item():.4f}"})

    # 记录每个 epoch 的平均 Loss
    loss_history_eps.append(epoch_loss_eps / len(dataloader))
    loss_history_x.append(epoch_loss_x / len(dataloader))

# ==========================================
# 5. 绘制 Loss 曲线对比图 (报告重要素材)
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS+1), loss_history_eps, label='Target: Noise ($\epsilon$-prediction)', marker='o', color='red')
plt.plot(range(1, EPOCHS+1), loss_history_x, label='Target: Clean Image ($x$-prediction)', marker='s', color='blue')
plt.title('Training Loss Comparison: $\epsilon$-prediction vs $x$-prediction', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# 保存图表，供报告使用
plt.savefig('/content/drive/MyDrive/Denoise_Experiment/prediction_comparison.png', dpi=300)
plt.show()

print("✅ 训练完成！对比图已生成并保存到网盘。")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import numpy as np
import umap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import time

# ==========================================
# 0. 基础设置与设备准备
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 启动【真·终极修正版】联合去噪实验，计算设备: {device}")

# 提升样本量至 10000，让 UMAP 构建的流形更加平滑致密
N_TRAIN = 10000
np.random.seed(42)
indices = np.random.choice(len(X_all), N_TRAIN, replace=False)

X_train = X_all[indices]
y_noisy = y_noisy_all[indices]
y_clean = y_clean_all[indices]

# 【核心 Bug 修复】：先还原 32x32x3，再用 permute 切换通道位置 (N, 3, 32, 32)
# 这样保证输入的绝对是一张正常的图片，而不是雪花噪点！
X_tensor = torch.tensor(X_train, dtype=torch.float32).reshape(-1, 32, 32, 3).permute(0, 3, 1, 2)

# ==========================================
# 1. 阶段一：提取高级特征
# ==========================================
print("\n[Phase 1] 将图像放大至 224x224，提取 ResNet18 高级特征...")
resnet = models.resnet18(weights='DEFAULT').to(device)
resnet.fc = nn.Identity()
resnet.eval()

transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

features_list = []
with torch.no_grad():
    dataset_feat = TensorDataset(X_tensor)
    loader_feat = DataLoader(dataset_feat, batch_size=128)
    for batch in loader_feat:
        inputs = transform_pipeline(batch[0]).to(device)
        feat = resnet(inputs)
        features_list.append(feat.cpu().numpy())
F_train = np.vstack(features_list)

# ==========================================
# 2. 阶段二：UMAP 流形梳理与伪标签生成
# ==========================================
print("\n[Phase 2] 正在运行 UMAP 提取流形先验 (处理1万样本约需30秒)...")
start_time = time.time()

umap_model = umap.UMAP(n_neighbors=20, min_dist=0.1, metric='cosine', random_state=42)
F_umap = umap_model.fit_transform(F_train)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(F_umap, y_noisy)
y_pseudo = knn.predict(F_umap)

umap_acc = accuracy_score(y_clean, y_pseudo)
raw_acc = accuracy_score(y_clean, y_noisy)
print(f"-> 原始极端噪声准确率: {raw_acc:.2%}")
print(f"-> 特征流形 UMAP 平滑准确率: {umap_acc:.2%} (耗时 {time.time()-start_time:.1f}s)")

# ==========================================
# 3. 阶段三：轻量化 JiT 联合拉回训练
# ==========================================
class JiT_Manifold_Net(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=256, num_classes=10):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, (img_size//patch_size)**2, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=embed_dim*2, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.decoder = nn.Linear(embed_dim, in_chans * patch_size * patch_size)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape
        x_emb = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
        features = self.transformer(x_emb)

        x_recon = self.decoder(features).transpose(1, 2).reshape(B, C, H//self.patch_size, W//self.patch_size, self.patch_size, self.patch_size)
        x_recon = x_recon.permute(0, 1, 2, 4, 3, 5).reshape(B, C, H, W)
        cls_logits = self.classifier(features.mean(dim=1))

        return cls_logits, x_recon

print("\n[Phase 3] 启动 JiT 流形联合拉回训练...")
y_pseudo_tensor = torch.tensor(y_pseudo, dtype=torch.long)
dataset_train = TensorDataset(X_tensor, y_pseudo_tensor)
dataloader = DataLoader(dataset_train, batch_size=64, shuffle=True)

model = JiT_Manifold_Net().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_cls = nn.CrossEntropyLoss()
criterion_x = nn.MSELoss()

ALPHA = 3.0  # 极强流形回归约束

model.train()
EPOCHS = 20
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        logits, x_recon = model(batch_x)
        loss_cls = criterion_cls(logits, batch_y)
        loss_x = criterion_x(x_recon, batch_x)

        loss = loss_cls + ALPHA * loss_x
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Total Loss: {total_loss/len(dataloader):.4f} | Recon Loss: {loss_x.item():.4f}")

model.eval()
with torch.no_grad():
    all_preds = []
    loader_eval = DataLoader(dataset_train, batch_size=256)
    for batch_x, _ in loader_eval:
        logits, _ = model(batch_x.to(device))
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

final_acc = accuracy_score(y_clean, all_preds)

print("\n" + "="*45)
print("🏆 实验三最终结果总结 🏆")
print("="*45)
print(f"1. 原始极度噪声准确率 (Baseline)       : {raw_acc:.2%}")
print(f"2. 深度特征 UMAP 提取先验准确率       : {umap_acc:.2%}")
print(f"3. UMAP + JiT(x-预测) 联合去噪准确率  : {final_acc:.2%}")
print("="*45)